# 07 — Explainability with SHAP

Uses SHAP (SHapley Additive exPlanations) to explain the XGBoost model's
predictions and understand which sensor features drive failure alerts.

## What this notebook does

1. Loads the trained XGBoost model and the test feature set.
2. Computes SHAP values for all test samples.
3. Plots:
   - **Summary plot** — global feature importance via SHAP
   - **Beeswarm plot** — distribution of SHAP values per feature
   - **Waterfall plot** — explanation for a single high-risk prediction

**Prerequisite:** Run notebooks 03 and 04 first.

## Imports

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import shap

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "saved_models"
FIGURES_DIR   = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DROP_COLS = ["timestamp", "machine_id", "failure_within_48h", "time_to_failure"]

## 1 — Load Model & Test Data

In [ ]:
xgb = joblib.load(MODELS_DIR / "xgboost.pkl")

df_test = pd.read_csv(PROCESSED_DIR / "test_engineered.csv", parse_dates=["timestamp"])
drop = [c for c in DROP_COLS if c in df_test.columns]
X_test = df_test.drop(columns=drop)
y_test = df_test["failure_within_48h"]

# Use a 2000-sample subset for speed
X_sample = X_test.sample(2000, random_state=42)
print(f"Sample shape: {X_sample.shape}")

## 2 — Compute SHAP Values

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer(X_sample)
print("SHAP values computed.")

## 3 — Summary Plot

In [ ]:
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
plt.title("SHAP Feature Importance — XGBoost")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_summary_bar.png", dpi=120)
plt.show()

## 4 — Beeswarm Plot

In [ ]:
shap.plots.beeswarm(shap_values, show=False)
plt.title("SHAP Beeswarm — XGBoost")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_beeswarm.png", dpi=120)
plt.show()

## 5 — Waterfall Plot (single high-risk prediction)

In [ ]:
# Find the sample with the highest predicted failure probability
probs = xgb.predict_proba(X_sample)[:, 1]
high_risk_idx = probs.argmax()
print(f"Highest predicted probability: {probs[high_risk_idx]:.3f}")

shap.plots.waterfall(shap_values[high_risk_idx], show=False)
plt.title("SHAP Waterfall — Highest Risk Sample")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_waterfall_high_risk.png", dpi=120)
plt.show()

## Next Steps

→ Use `src/predict.py` for product-style CLI predictions:
```
python src/predict.py --machine Machine_3 --model xgboost
```